In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.filters import threshold_otsu
from skimage.color import label2rgb

# --- 1. Auto-Discover File Path ---
print("Searching for the 'stage1-train' dataset folder...")
dataset_dir = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'stage1-train' in dirs:
        dataset_dir = os.path.join(root, 'stage1-train')
        break

if not dataset_dir:
    print("ERROR: Could not find 'stage1-train' in Kaggle inputs.")
    print("Please ensure the Data Science Bowl 2018 dataset is added to your notebook.")
else:
    print(f"SUCCESS! Found dataset at: {dataset_dir}")

    # --- 2. Load and Preprocess the Image ---
    # Get the first folder in the training directory
    sample_folders = sorted(os.listdir(dataset_dir))
    first_sample_id = sample_folders[0]
    
    # Construct the path to the actual PNG image
    image_file_path = os.path.join(dataset_dir, first_sample_id, "images", f"{first_sample_id}.png")

    # Read image and convert color spaces
    raw_bgr = cv2.imread(image_file_path)
    original_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    grayscale = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)
    
    # Apply slight Gaussian blur to smooth out noise
    smoothed_gray = cv2.GaussianBlur(grayscale, (5, 5), 0)

    # --- 3. Segmentation Pipeline ---
    # Otsu's Thresholding to create a binary mask (foreground vs background)
    otsu_limit = threshold_otsu(smoothed_gray)
    binary_mask = smoothed_gray > otsu_limit

    # Calculate Distance Transform
    dist_transform = ndi.distance_transform_edt(binary_mask)

    # METHOD A: Watershed WITHOUT Markers (Baseline/Over-segmented)
    labels_uncontrolled = watershed(-dist_transform, mask=binary_mask)

    # METHOD B: Marker-Controlled Watershed (Corrected)
    # 1. Find the highest peaks in the distance map using a 25x25 window
    peak_coordinates = peak_local_max(dist_transform, footprint=np.ones((25, 25)), labels=binary_mask)
    
    # 2. Map these peaks to a boolean mask
    seed_mask = np.zeros_like(dist_transform, dtype=bool)
    seed_mask[tuple(peak_coordinates.T)] = True
    
    # 3. Give each peak a unique numerical label
    seed_markers, _ = ndi.label(seed_mask)
    
    # 4. Run the watershed algorithm starting ONLY from our seed markers
    labels_controlled = watershed(-dist_transform, seed_markers, mask=binary_mask)

    # --- 4. Visualization Layout ---
    # Using modern subplots instead of older MATLAB-style numbering
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    ax = axes.ravel() # Flatten to a 1D array for easy indexing

    # Top Row
    ax[0].imshow(original_rgb)
    ax[0].set_title("Original Image")
    ax[0].axis('off')

    ax[1].imshow(binary_mask, cmap='gray')
    ax[1].set_title("Binary Mask")
    ax[1].axis('off')

    ax[2].imshow(dist_transform, cmap='jet')
    ax[2].set_title("Distance Transform")
    ax[2].axis('off')

    # Bottom Row
    ax[3].imshow(label2rgb(labels_uncontrolled, image=original_rgb, bg_label=0))
    ax[3].set_title("Watershed WITHOUT Markers")
    ax[3].axis('off')

    ax[4].imshow(label2rgb(labels_controlled, image=original_rgb, bg_label=0))
    ax[4].set_title("Marker-Controlled Watershed")
    ax[4].axis('off')

    # Hide the empty 6th subplot
    ax[5].axis('off')

    plt.tight_layout()
    plt.show()